# Text-to-SVG Generation — Training Notebook
**Course:** CS-GY 9223 Deep Learning — Spring 2026, NYU Tandon  
**Competition:** NYU Deep Learning Spring 2026 Kaggle Contest  
**Task:** Generate valid SVG code from natural language text prompts

## Training Overview

This notebook covers the full training pipeline for our text-to-SVG
generation model. It should be run in Google Colab with GPU enabled.

The notebook is structured as follows:

1. **Setup and Imports** — load libraries and set random seeds
2. **Data Loading and Cleaning** — download and preprocess training data
3. **Data Preparation** — tokenize prompts and SVG strings
4. **Model Setup** — load pretrained model and configure for fine-tuning
5. **Training** — run training loop and save best checkpoints
6. **Validation** — evaluate on held-out validation set

**Important notes:**
- Runtime: Runtime → Change Runtime Type → GPU before running
- All weights are saved to Google Drive to persist across sessions
- Random seed is fixed at 42 for reproducibility

## Key Findings from EDA

Based on our exploratory data analysis the following properties of
the dataset inform our training decisions:

- Prompts average ~20 words — T5-base or CLIP text encoder is sufficient
- SVGs are short and simple — majority under 2,000 characters and 50 paths
- Dataset is icon-focused and monochromatic — model does not need to
  handle complex scenes
- Canvas sizes vary in training data — 256x256 normalization handled
  at inference time
- Final clean training set after preprocessing: 45,130 rows

In [1]:
# ================================
# Section 1: Setup and Imports
# ================================

# core libraries
import os
import re
import random
import numpy as np
import pandas as pd

# deep learning
import torch

# set random seeds — required for reproducibility
# competition guide requires fixed random seeds in all code
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f"Random seed set to: {SEED}")

Random seed set to: 42


In [2]:
# verify GPU is available before doing anything
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected — go to Runtime → Change Runtime Type → T4 GPU")

GPU available: False


In [3]:
# ================================
# Section 2: Kaggle Setup
# ================================

from google.colab import files
files.upload()  # upload kaggle.json when prompted
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install kaggle

Saving kaggle.json to kaggle.json


In [4]:
# download competition data
!kaggle competitions download -c dl-spring-2026-svg-generation



100% 32.4M/32.4M [00:00<00:00, 45.3MB/s]



FileNotFoundError: [Errno 2] No such file or directory: 'YOUR-COMPETITION-NAME.zip'

In [5]:
# unzip
import zipfile
with zipfile.ZipFile('dl-spring-2026-svg-generation.zip', 'r') as z:
    z.extractall('data/')

print(os.listdir('data/'))

['test.csv', 'train.csv', 'sample_submission.csv']


In [6]:
# load raw data
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Raw training rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")

Raw training rows: 50000
Test rows: 1000


In [10]:
# ================================
# Section 2: Cleaning
# ================================

# compute prompt length
train_df['prompt_len'] = train_df['prompt'].str.split().str.len()

# define filters — same as applied in EDA notebook
non_ascii = train_df['prompt'].str.contains('[^\x00-\x7F]')
very_short = train_df['prompt_len'] < 3

# report before cleaning
print("=== BEFORE CLEANING ===")
print(f"Total rows: {len(train_df)}")
print(f"Non-ASCII prompts: {non_ascii.sum()}")
print(f"Under 3 word prompts: {very_short.sum()}")

# apply cleaning
train_clean = train_df[~(non_ascii | very_short)].reset_index(drop=True)
train_clean = train_clean.drop(columns=['prompt_len'])

# confirm final count
print("\n=== AFTER CLEANING ===")
print(f"Clean training rows: {len(train_clean)}")
assert len(train_clean) == 49195, f"Expected 49195 rows but got {len(train_clean)}"
print("Assertion passed — correct number of rows confirmed")

=== BEFORE CLEANING ===
Total rows: 50000
Non-ASCII prompts: 794
Under 3 word prompts: 20

=== AFTER CLEANING ===
Clean training rows: 49195
Assertion passed — correct number of rows confirmed


#  Section 3 — Data Preparation (tokenization)

# Section 4 — Model Setup (load T5)

# Section 5 — Training Loop

# Section 6 — Validation